# Load Datasets
### Unsupervised Learning (Wine Dataset)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
!pip install -q fcmeans
from fcmeans import FCM

## Part A (Exploratory Data Analysis)

In [ ]:
data = load_wine()
X, y = data.data, data.target

print('Shape:', X.shape)
print('Features:', list(data.feature_names))
print('True classes (for reference only):', list(data.target_names))

In [ ]:
df = pd.DataFrame(X, columns=data.feature_names)
df.describe().loc[['mean','std','min','max']].T

In [ ]:
counts = np.bincount(y)
print(' | '.join(f'{name}: {counts[i]}' for i, name in enumerate(data.target_names)))
print(f'Total samples: {len(y)}')

plt.figure(figsize=(5,4))
plt.bar(data.target_names, counts, color=['salmon','steelblue','seagreen'])
plt.title('True Class Distribution (Reference Only)')
plt.xlabel('Wine Type'); plt.ylabel('Count')
for i,v in enumerate(counts): plt.text(i, v+1, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## Part B (Elbow Method – Optimal k)

In [ ]:
inertias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(list(K_range), inertias, marker='o', color='steelblue')
plt.xlabel('Number of Clusters (k)'); plt.ylabel('Inertia')
plt.title('Elbow Method – K-Means on Wine Dataset')
plt.xticks(list(K_range))
plt.tight_layout(); plt.show()

print('Inertia by k:')
for k, inertia in zip(K_range, inertias):
    print(f'  k={k}: {inertia:.2f}')

## Part C (K-Means Clustering – Different k Values)

In [ ]:
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sizes = np.bincount(labels)
    print(f'k={k}  |  Cluster sizes: {dict(enumerate(sizes))}  |  Inertia: {km.inertia_:.2f}')

In [ ]:
k_opt = 3
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)

km_df = pd.DataFrame({'Cluster': kmeans_labels})
print(km_df['Cluster'].value_counts().sort_index())

## Part D (Agglomerative Clustering & Dendrogram)

In [ ]:
linked = linkage(X, method='ward')

plt.figure(figsize=(12,5))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=8)
plt.title('Dendrogram – Hierarchical Clustering (Ward Linkage)')
plt.xlabel('Sample Index (or Cluster Group)'); plt.ylabel('Distance')
plt.tight_layout(); plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
agg_labels = agg.fit_predict(X)

print('Agglomerative cluster sizes:')
print(pd.Series(agg_labels).value_counts().sort_index())

**Dendrogram interpretation:** Each vertical line merges two clusters. The **height** shows how far apart merged groups are — taller merges mean more distinct clusters. A long vertical line before a merge suggests a good cut point. Here, cutting around **k = 3** aligns with the three wine types in the dataset.

## Part E (Fuzzy C-Means Clustering)

In [ ]:
fcm = FCM(n_clusters=3, random_state=42)
fcm.fit(X)
fcm_labels = fcm.predict(X)
membership = fcm.u

print('Fuzzy C-Means cluster sizes:')
print(pd.Series(fcm_labels).value_counts().sort_index())
print()
print(f'Membership matrix shape: {membership.shape}  (samples × clusters)')
print('Sample 0 membership scores:', np.round(membership[0], 4))

In [ ]:
max_membership = membership.max(axis=1)
print(f'Mean max membership : {max_membership.mean():.4f}')
print(f'Min  max membership : {max_membership.min():.4f}')
print(f'Samples with max membership < 0.6: {(max_membership < 0.6).sum()}')

## Part F (Bonus – DBSCAN, Default Parameters)

In [ ]:
dbscan = DBSCAN()
db_labels = dbscan.fit_predict(X)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

print(f'DBSCAN clusters found : {n_clusters}')
print(f'Noise points (label=-1): {n_noise}')
print()
print('Label counts:')
print(pd.Series(db_labels).value_counts().sort_index())

## Part G (Short Interpretation)

In [ ]:
print(f'Wine dataset: {X.shape[0]} samples, {X.shape[1]} features (no scaling applied).')
print(f'Elbow method suggests k≈3; K-Means (k=3) inertia: {kmeans.inertia_:.2f}.')
print(f'Agglomerative (k=3) and Fuzzy C-Means produced {len(set(agg_labels))} and {len(set(fcm_labels))} clusters.')
print(f'DBSCAN (default eps=0.5, min_samples=5): {n_clusters} clusters, {n_noise} noise points.')